# Text to 3D AI Model - Colab + ngrok

This notebook runs the GitHub project on a Colab GPU and exposes the FastAPI frontend/backend through ngrok.

## 1. Clone the GitHub repository

In [13]:
REPO_URL = "https://github.com/LucyAlex12/Text_to_3d_ai_model.git"

!rm -rf Text_to_3d_ai_model
!git clone {REPO_URL}
%cd Text_to_3d_ai_model

Cloning into 'Text_to_3d_ai_model'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 77 (delta 24), reused 64 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 30.28 MiB | 17.14 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Text_to_3d_ai_model/Text_to_3d_ai_model


In [14]:
import sys
import os

TRIPOSR_PATH = "/content/Text_to_3d_ai_model/TripoSR"

if TRIPOSR_PATH not in sys.path:
    sys.path.append(TRIPOSR_PATH)

print("Added TripoSR to sys.path")
print(sys.path[-1])

Added TripoSR to sys.path
/content/Text_to_3d_ai_model/TripoSR


## 2. Install dependencies

If Colab asks you to restart the runtime after installs, restart it, then rerun the cells from the top.

In [1]:
# Clean conflicting packages first
!pip uninstall -y cupy cupy-cuda12x cupy-cuda11x numpy pymatting rembg

# Stable numpy for Colab + torch ecosystem
!pip install numpy==1.26.4

# Install backend dependencies
!pip install rembg==2.0.59 pymatting==1.1.12
!pip install fastapi uvicorn python-multipart pyngrok pillow
!pip install transformers diffusers accelerate safetensors
!pip install trimesh xatlas pygltflib

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: PyMatting 1.1.12
Uninstalling PyMatting-1.1.12:
  Successfully uninstalled PyMatting-1.1.12
Found existing installation: rembg 2.0.59
Uninstalling rembg-2.0.59:
  Successfully uninstalled rembg-2.0.59
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, which is not installed.
pylibcugraph-cu12 26.2.0 requires cupy-cuda12x>=13.6.0, which is not installed.
cudf-cu12 26.2.1 requires cupy-cuda12x>=13.6.0, which is not installed.
dask-cudf-cu12 26.2.1 requires cupy-cuda12x>=13.6.0, which is n

  Using cached rembg-2.0.59-py3-none-any.whl.metadata (17 kB)
  Using cached PyMatting-1.1.12-py3-none-any.whl.metadata (7.4 kB)
Using cached rembg-2.0.59-py3-none-any.whl (39 kB)
Using cached PyMatting-1.1.12-py3-none-any.whl (52 kB)


## 3. Download TripoSR weights

`TripoSR/model.ckpt` is large, so the notebook downloads it from Hugging Face instead of storing it in Git.

In [2]:
from pathlib import Path
from huggingface_hub import hf_hub_download

Path("TripoSR").mkdir(exist_ok=True)

for filename in ["config.yaml", "model.ckpt"]:
    hf_hub_download(
        repo_id="stabilityai/TripoSR",
        filename=filename,
        local_dir="TripoSR"
    )

print("TripoSR checkpoint ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.yaml:   0%|          | 0.00/987 [00:00<?, ?B/s]

model.ckpt:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

TripoSR checkpoint ready


## 4. Configure ngrok

Create an ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken.

The next cell uses `getpass()` instead of Colab Secrets because Colab's secret vault can fail with `Failed to fetch` / `await connected: disconnected` errors.

Optional: if you reserved an ngrok static domain, enter it when prompted, for example `your-name.ngrok-free.app`.

In [3]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ").strip()
NGROK_STATIC_DOMAIN = input("Optional ngrok static domain, or press Enter: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("ngrok authtoken is required")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

if NGROK_STATIC_DOMAIN:
    print("ngrok configured with static domain:", NGROK_STATIC_DOMAIN)
else:
    print("ngrok configured with a temporary URL")

Paste your ngrok authtoken: ··········
Optional ngrok static domain, or press Enter: 
ngrok configured with a temporary URL


## 5. Start backend and expose the app

Open the printed ngrok URL in a normal browser tab. Do not use Colab's iframe preview.

In [4]:
!rm -rf /content/Text_to_3d_ai_model/TripoSR

!git clone https://github.com/VAST-AI-Research/TripoSR.git \
/content/Text_to_3d_ai_model/TripoSR

Cloning into '/content/Text_to_3d_ai_model/TripoSR'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 161 (delta 63), reused 42 (delta 42), pack-reused 66 (from 1)
Receiving objects: 100% (161/161), 36.71 MiB | 18.39 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [5]:
import sys

sys.path.append(
    "/content/Text_to_3d_ai_model/TripoSR"
)

In [10]:
%cd /content/Text_to_3d_ai_model

!ls

/content/Text_to_3d_ai_model
1778172473.png	LICENSE      README.md		  Text_to_3D_Colab.ipynb
api.py		pipeline.py  requirements.txt	  TripoSR
index.html	__pycache__  Text_to_3d_ai_model


In [14]:
import sys

sys.path.append("/content/Text_to_3d_ai_model/TripoSR")

import api

Using device: cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/Text_to_3d_ai_model/TripoSR/config.yaml'

In [9]:
import os
import subprocess
import time
import requests
from pyngrok import ngrok

os.environ["IMAGE_MODEL_KIND"] = "sd15"
os.environ["SDXL_WIDTH"] = "512"
os.environ["SDXL_HEIGHT"] = "512"
os.environ["SDXL_STEPS"] = "20"
os.environ["SDXL_GUIDANCE_SCALE"] = "7.0"
os.environ["TRIPOSR_MAX_MC_RESOLUTION"] = "256"

ngrok.kill()

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting backend. This can take a few minutes while models load...")

for _ in range(240):
    if server.poll() is not None:
        remaining = server.stdout.read() if server.stdout else ""
        raise RuntimeError("Backend stopped before it was ready. Logs:\n" + remaining[-4000:])
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("Backend is ready")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not become ready. Run the logs cell below.")

if NGROK_STATIC_DOMAIN:
    tunnel = ngrok.connect(8000, "http", domain=NGROK_STATIC_DOMAIN)
else:
    tunnel = ngrok.connect(8000, "http")

public_url = tunnel.public_url

print("\nOpen this public app URL:")
print(public_url)
print("\nKeep this Colab runtime running while using the app.")
print("If ngrok shows a browser warning page, click through once. The app also sends the ngrok-skip-browser-warning header for API/model requests.")

Starting backend. This can take a few minutes while models load...


RuntimeError: Backend stopped before it was ready. Logs:
ERROR:    Error loading ASGI app. Could not import module "api".


## Optional: view backend logs

Run this only if the app fails or you want to monitor generation logs.

In [ ]:
while True:
    line = server.stdout.readline()
    if not line:
        break
    print(line, end="")